# aw_07_b4 — Stage B4: P1 champion → PlayWorld SFT (Track B Phase-2, §5.1 / RQ1)

**Parent = P1 champion = B1v2** (`20260807-225109--b1-general-sft-v2--s42--e6e83b`,
sha256:747e5757...; §6 selection recorded in aw_06_b3). B3 (P1 DPO) was a null
transfer result, so the two-stage pipeline proceeds from the SFT champion.

**Design.** B4 is the RQ1 treatment arm: identical PlayWorld SFT budget to A1
(same data seed 1042, same recipe hyperparameters, 2 epochs full budget — NOT
the 200-step probe), differing from A1 in exactly one variable: initialization
(B1v2 P1 adapter vs base). Primary readout: `f_b4_analysis` B4 vs A1 on the
frozen suites (eval-ID + OOD splits), plus B4 vs A2 as the strongest Track-A
comparator.

Cell order: fetch champion → `a_b4_data` → `b_b4_train` → `c_b4_eval` →
`x09f_run_audit` → `f_b4_analysis`.

**Stage record (2026-08-13):** run `20260812-235057--b4-playworld-sft-from-p1--s42--72f4f6`
(parent sha 747e5757..., output sha 378ba470..., train_loss profile healthy, 2ep/250 steps).
Eval run `20260813-000613--eval-playworld--s42--c49f3a`, freeze_fingerprint 3cdcbc30 MATCH,
truncation/runaway 0.0. **RQ1 primary: B4 ≫ A1 on ALL 10 suite×metric cells (p=.0001):**
eval-ID pass +.200 (.3867 vs .1867), template-OOD +.157, comp-OOD +.160, rule-OOD +.127,
adversarial +.250. B4 also ≫ A2 on all cells (p=.0001).

**⚠ OPEN AUDIT (x15): sft_fingerprint drift.** a_b4_data produced
sha256:2764e797... but the aw_05/aw_06 probe builds produced sha256:050f94b2...
(identical CLI/seed; prompt_fingerprint cc2aef0d... matched). The divergence is
confined to the SFT serialization. Run `scripts/x15_sft_data_diff.py` against the
frozen A1-era artifact before treating the RQ1 result as final: multiset-identical
⇒ caveat only; content-diverged ⇒ retrain B4 on the frozen artifact.


**x15 VERDICT (2026-08-13): CONTENT DIVERGED — protocol deviation D1 (v1.3).**
53/2000 records (2.65%) differ between the A1-era frozen artifact and the aw_07
rebuild; prompts identical (cc2aef0d stable across ALL builds), divergence
confined to the oracle demonstration's target choice among equally-optimal BFS
paths → the generator is deterministic only per code commit. All PlayWorld
*evaluations* remain valid (freeze_fingerprint 3cdcbc30 matched on every eval,
incl. A1/A2/probes/B4) and B-track probes all used the same 050f94b2 build, so
B1–B3 conclusions stand. Only the B4-vs-A1 single-variable claim is confounded
(~2.65% oracle tie-break delta). Remediation: x16 resolves the artifact A1
actually trained on (lineage fingerprint match) → retrain **B4v2** on that
frozen artifact → re-run eval/audit/analysis below. v1 B4 run is retained as
the documented deviation baseline.


**x16 RESULT (2026-08-13): both matches NULL.** A1 (f8ea5785) and B4 (2764e797)
each trained on unfrozen in-session builds; none of the 3 persisted HF artifacts
(cc2bd418 / c0cfa17e / 042eb078) matches either. A1's exact bytes are
unrecoverable ⇒ the comparison cannot be restored by retraining B4 alone.
**Remediation (protocol v1.3): pin canonical =
`m97j/aw-playworld:train/v1/playworld_sft.jsonl` (042eb078, current frozen
commit) and retrain BOTH arms — A1v2 and B4v2 — on it (original recipes).**
RQ1 headline rebases to B4v2 vs A1v2; v1 runs stay as deviation baselines.


**v2 STAGE RECORD (CLOSED 2026-08-14).** Canonical artifact pinned
(0f60f9b0 file-sha / 042eb078 lineage-fp). a1v2 run 269d0e (adapter 70c2ecb9),
b4v2 run c56ed2 (adapter d4fcacdd), both on identical bytes; evals a27857/7308ee,
freeze 3cdcbc30 matched. **RQ1 canonical: B4v2 ≫ A1v2 on all 10 cells (p=.0001)** —
eval-ID pass .3933 vs .1667 (+.227), comp-OOD +.167, adversarial +.263.
Sensitivity a1v2-vs-a1v1: all 10 cells n.s. (|Δ|≤.02) → the D1 tie-break drift is
behaviorally benign; the v1 headline replicated under clean provenance.
Secondary observation for the report: A-track arms exhibit the known base-init
termination pathology at eval (a1v2 truncation .994 / runaway .888; scores remain
valid because the verifier parses the leading action block — legal_action_rate
.99 on adversarial), while B4v2 stops cleanly (trunc/runaway 0.0), inheriting the
trained <|im_end|> from B1v2 (§13 v1.2). Two-stage therefore also fixes
termination "for free" — report as a mechanistic advantage, not a confound
(pass verdicts are termination-insensitive).
b4v2-vs-b4v1 sensitivity errored on a missing fetch (see fixed cell) — rerun.


## Archive status — publication completed

This notebook combines historical v1 training/evaluation records with the completed
September 2026 publication session. The publication was executed in **web Colab**;
the setup header still refers to the earlier VS Code workflow.

- Published artifact: **FP32 standalone safetensors**, original adapter under `adapter/`.
- Immutable release: [m97j/aw-qwen3-8b-v1@5b71e3e4](https://huggingface.co/m97j/aw-qwen3-8b-v1/tree/5b71e3e41d3c7630b4c6320aa67148ab653ac96d).
- Publication receipt: [published.json](evidence/champion_release_20260923/published.json).
- Evaluation limitation: FP32 batch 4 and BF16 batch 256; each has its own paired
  adapter control. These are not matched cross-precision results.

**Execution archive, not a Run All workflow.** The Colab runtime has ended.
Do not rerun training or publication to clean up outputs. Review evidence and
choose a new output path before any future experiment. Original code, execution
counts and outputs are retained, including failures and the successful retry.
Explanatory Markdown was reorganized; Korean comments and unexecuted diagnostic error strings were translated to English. Outputs were not edited.


In [ ]:
# @title common header — VS Code notebook / Colab kernel
import os
import sys
import json
import subprocess
from pathlib import Path
from getpass import getpass

from google.colab import userdata

# Retained for historical training reproduction; not needed for publication.
# os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
if not os.environ.get("HF_TOKEN"):
    try:
        os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    except Exception:
        os.environ["HF_TOKEN"] = getpass("HF token (hidden; write access for publication): ")

def logged_command(argv, *, cwd=None):
    from datetime import datetime, timezone
    from uuid import uuid4

    log_dir = Path("/content/axiom-world-release-logs")
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
                          + "-" + uuid4().hex[:8] + ".log")
    env = os.environ.copy()
    env.update(PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8",
               HF_HUB_DISABLE_PROGRESS_BARS="1", PIP_PROGRESS_BAR="off")
    if cwd is not None:
        env["PYTHONPATH"] = str(Path(cwd) / "src")
    print("[start]", " ".join(map(str, argv)), flush=True)
    print("[log]", log_path, flush=True)
    with log_path.open("x", encoding="utf-8") as log:
        log.write("COMMAND " + json.dumps(list(map(str, argv))) + "\n")
        log.flush()
        with subprocess.Popen(argv, cwd=cwd, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True,
                              encoding="utf-8", errors="replace", bufsize=1) as process:
            try:
                for line in process.stdout:
                    log.write(line)
                    log.flush()
                    print(line, end="", flush=True)
                code = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                log.write("\n[interrupted]\n")
                raise
        log.write(f"\n[exit] {code}\n")
    print(f"[exit] {code}; log={log_path}", flush=True)
    if code:
        raise RuntimeError(f"Command failed (exit={code}). See the traceback above or {log_path}")


def verify_release_checkout():
    expected_root = Path("/content/axiom-world-fp32")
    expected_revision = "e2a66ab73c4054e59339b8870bf437a0b586141f"
    if REPO_ROOT != expected_root or CODE_REVISION != expected_revision:
        raise RuntimeError("Stale notebook/kernel configuration: rerun the updated common header")
    actual = subprocess.check_output(
        ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
    if actual != expected_revision:
        raise RuntimeError(f"Wrong execution checkout: {actual}; expected {expected_revision}")
    subprocess.run(["git", "-C", str(REPO_ROOT), "diff", "--quiet", "HEAD", "--",
                    "src/axiom_world/models/champion_release.py",
                    "scripts/publish/v1/publish_champion.py",
                    "scripts/publish/v1/compare_champion.py",
                    "scripts/publish/v1/derive_bf16_candidate.py",
                    "scripts/publish/v1/review_precision.py",
                    "scripts/publish/v1/finalize_fp32_audit.py"], check=True)
    print(f"[preflight] FP32 code={actual}; workspace={REPO_ROOT}", flush=True)


def release_command(*args):
    verify_release_checkout()
    logged_command([sys.executable, "-u", *args], cwd=REPO_ROOT)

CODE_REVISION = "e2a66ab73c4054e59339b8870bf437a0b586141f"  # published FP32 implementation
REPO_ROOT = Path("/content/axiom-world-fp32")  # Colab filesystem, not the Windows workspace
if not REPO_ROOT.exists():
    logged_command(["git", "clone", "https://github.com/m97j/axiom-world.git", str(REPO_ROOT)])
else:
    existing = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
    if existing != CODE_REVISION:
        if existing not in {"5298f52b3f60aaed8cb003e4284d53ea53cc263f", "852edecb4f8a8035e0798675f61aa3b3594ba80f", "1daef869d312dfc91a6d400d56e7851a421eeee8", "bf19c472b65ccd4fd5af848ce3f8ef1a8b189a46", "882c84a3a954f084757967f62af6ab722fb441ca"}:
            raise RuntimeError("Unexpected checkout; inspect it before changing revisions")
        # Preserve runs/ and all merge evidence. Refuse changes to tracked files.
        logged_command(["git", "-C", str(REPO_ROOT), "diff", "--quiet", "HEAD"])
        logged_command(["git", "-C", str(REPO_ROOT), "fetch", "origin"])
logged_command(["git", "-C", str(REPO_ROOT), "checkout", "--detach", CODE_REVISION])
os.chdir(REPO_ROOT)
print("Execution code commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
logged_command([sys.executable, "-u", "-m", "pip", "install", "-e", ".",
                "-r", "requirements/colab-g4.lock.txt"], cwd=REPO_ROOT)


RELEASE_OUTPUT = "runs/champion-v1-release-fp32"  # preserve the completed FP32 source
COMPARISON_OUTPUT = "runs/champion-v1-comparison"
BF16_RELEASE_OUTPUT = "runs/champion-v1-release-bf16-candidate-native-buffers"
BF16_COMPARISON_OUTPUT = "runs/champion-v1-comparison-bf16-native-buffers"

# Editable installs in a child process may not refresh this running kernel.
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))


[start] git -C /content/axiom-world-fp32 diff --quiet HEAD
[log] /content/axiom-world-release-logs/20260923T170821Z-2943bdee.log
[exit] 0; log=/content/axiom-world-release-logs/20260923T170821Z-2943bdee.log
[start] git -C /content/axiom-world-fp32 fetch origin
[log] /content/axiom-world-release-logs/20260923T170821Z-46bd13f8.log
From https://github.com/m97j/axiom-world
   c34bbbe..9f43d7c  main       -> origin/main
[exit] 0; log=/content/axiom-world-release-logs/20260923T170821Z-46bd13f8.log
[start] git -C /content/axiom-world-fp32 checkout --detach e2a66ab73c4054e59339b8870bf437a0b586141f
[log] /content/axiom-world-release-logs/20260923T170821Z-c05271fb.log
Previous HEAD position was 5298f52 Preserve native runtime buffers when deriving BF16 weights
HEAD is now at e2a66ab Disclose independent-control precision audits in FP32 release
[exit] 0; log=/content/axiom-world-release-logs/20260923T170821Z-c05271fb.log
Execution code commit: e2a66ab73c4054e59339b8870bf437a0b586141f
[start] /usr

In [ ]:
# @title fetch champion — materialize the B1v2 parent adapter + lineage sha
B1V2_RUN_ID = "20260807-225109--b1-general-sft-v2--s42--e6e83b"

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b1 --run-id {B1V2_RUN_ID}
b1v2_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

import json
b1v2_sha = json.load(open(f"runs/{B1V2_RUN_ID}/artifacts/lineage.json"))["output_adapter_sha256"]
print(b1v2_dir, b1v2_sha)


runs/20260807-225109--b1-general-sft-v2--s42--e6e83b/artifacts/final_adapter sha256:747e57574f811274d34852f8347c47c2ff90a86186f616f796856c7f01cae21d


In [ ]:
# @title a_b4_data — deterministic PlayWorld train data + frozen suites (leakage-gated)
# Same generator/seed as A1 (seed 1042, 5 families x 400). Verify the manifest
# sft_fingerprint matches the A1-era value sha256:050f94b2... — any mismatch
# breaks the single-variable design and MUST stop the stage.
!python scripts/build_training_data.py \
  --seed 1042 --scenarios-per-family 400 --output-dir data/train

!python scripts/build_eval_suites.py --episodes-per-suite 300
# freeze_manifest fingerprint MUST equal sha256:3cdcbc30c99e492c... (G3 freeze)


{
  "seed": 1042,
  "sft_records": 2000,
  "prompt_records": 2000,
  "unsolvable_dropped": 0,
  "sft_fingerprint": "sha256:2764e797e87f6d57fb2a77e1c29678f122b4f18752ab043ce5620ab4ee7f8f39",
  "prompt_fingerprint": "sha256:cc2aef0df4f5efda3db7ccfd43c76e40260935e9b7b55ee5760490e6a4304f14",
  "train_families": [
    "train-fam0",
    "train-fam1",
    "train-fam2",
    "train-fam3",
    "train-fam4"
  ],
  "eval_families_checked": [
    "eval_adversarial-fam0",
    "eval_comp_ood-fam0",
    "eval_comp_ood-fam1",
    "eval_id-fam0",
    "eval_id-fam1",
    "eval_id-fam2",
    "eval_rule_ood-fam0",
    "eval_rule_ood-fam1",
    "eval_template_ood-fam0",
    "eval_template_ood-fam1"
  ]
}
eval_id: 300 episodes -> data/eval_suites/eval_id.jsonl (sha256:aceeea727d2b9eaed...)
eval_template_ood: 300 episodes -> data/eval_suites/eval_template_ood.jsonl (sha256:13580a6cbf7a4e566...)
eval_comp_ood: 300 episodes -> data/eval_suites/eval_comp_ood.jsonl (sha256:444191a244dcd77d1...)
eval_rule_ood: 300

In [ ]:
# @title b_b4_train — PlayWorld SFT from the B1v2 parent (full A1 budget)
!python scripts/run_experiment.py \
  --config configs/experiments/b4_playworld_sft_from_p1.yaml \
  --parent-adapter-dir {b1v2_dir} \
  --override lineage.parent_run_id={B1V2_RUN_ID} \
  --override lineage.parent_adapter.repo_id=m97j/aw-runs-b1 \
  --override lineage.parent_adapter.revision=main \
  --override lineage.parent_adapter.sha256={b1v2_sha} \
  --override data.source.local_path=data/train/playworld_sft.jsonl \
  --hf-sync-repo m97j/aw-runs-b4


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
run_id: 20260812-235057--b4-playworld-sft-from-p1--s42--72f4f6
config.json: 100% 729/729 [00:00<00:00, 8.03MB/s]
tokenizer_config.json: 100% 9.68k/9.68k [00:00<00:00, 7.56MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 27.2MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 183MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 239MB/s]
model.safetensors.index.json: 100% 32.9k/32.9k [00:00<00:00, 195MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/7.96G [00:

In [ ]:
# @title c_b4_eval — B4 adapter on the frozen suites (canonical profile)
B4_RUN_ID = "20260812-235057--b4-playworld-sft-from-p1--s42--72f4f6"  # <- from b_b4_train "run_id: ..."

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4_RUN_ID}
b4_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {b4_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-b4


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Loading weights: 100% 399/399 [00:01<00:00, 339.19it/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
generate(batched): 100% 3/3 [00:33<00:00, 11.25s/it]
eval_adversarial: pass_rate={'mean': 0.8433, 

In [ ]:
# @title x09f_run_audit — termination regression check on the B4 eval (CPU)
B4_EVAL = "20260813-000613--eval-playworld--s42--c49f3a"  # <- eval run id from c_b4_eval

!python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4_EVAL} --kind eval
!python scripts/x09_termination_audit.py \
  --run-dirs runs/{B4_EVAL} --out runs/x09_run_audit_b4.json


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0% 0/8 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/451k [00:00<?, ?B/s]          
Reconstructing (incomplete total...): 100% 451k/451k [00:00<00:00, 3.53MB/s]
Reconstructing (incomplete total...):  50% 451k/899k [00:00<00:00, 3.53MB/s]
Reconstructing (incomplete total...):  50% 451k/902k [00:00<00:00, 3.53MB/s]
Reconstructing (incomplete total...): 100% 902k/903k [00:00<00:00, 3.53MB/s]

Fetching 8 files:  12% 1/8 [00:00<00:01,  5.16it/s]
Reconstructing (incomplete total...):  67% 903k/1.34M [00:00<00:00, 3.53MB/s]
Reconstructing (incomplete total...):  52% 903k/1.73M [00:00<00:00, 3.53MB/s]
Fetching 8 files: 100% 8/8 [00:00<00:00, 29.69it/s]
Download complete: 100% 2.17M/2.17M [00:00<00:00, 5.44MB/s]
Reconstruction complete: 100% 2.17M/2.17M [00:00<00:00, 5.44MB/s]             eval run materialized: 5 suite files, freeze_fingerprint=sha256:3cdcbc30c99e492c...
RUN_DI

In [ ]:
# @title f_b4_analysis — B4 vs A1 (RQ1 primary) and B4 vs A2
A1_EVAL = "20260801-063425--eval-playworld--s42--3bf440"
A2_EVAL = "20260803-011408--eval-playworld--s42--b6f315"  # <- canonical A2 eval run id (from aw_04_a2)

!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1_EVAL} --kind eval

!python scripts/run_analysis.py \
  --run-a runs/{B4_EVAL} --label-a b4-two-stage \
  --run-b runs/{A1_EVAL} --label-b a1-direct \
  --output runs/{B4_EVAL}/analysis_b4_vs_a1.json --hf-sync-repo m97j/aw-runs-b4

if A2_EVAL:
    !python scripts/fetch_run.py --repo m97j/aw-runs-a2 --run-id {A2_EVAL} --kind eval
    !python scripts/run_analysis.py \
      --run-a runs/{B4_EVAL} --label-a b4-two-stage \
      --run-b runs/{A2_EVAL} --label-b a2-dpo \
      --output runs/{B4_EVAL}/analysis_b4_vs_a2.json --hf-sync-repo m97j/aw-runs-b4


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0% 0/11 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/6.04k [00:00<?, ?B/s]         
Reconstructing (incomplete total...):  50% 6.04k/12.1k [00:00<00:00, 15.3kB/s]
Reconstructing (incomplete total...):  95% 12.1k/12.7k [00:00<00:00, 22.9kB/s]
Reconstructing (incomplete total...):   1% 12.7k/1.91M [00:00<01:22, 22.9kB/s]

Fetching 11 files:   9% 1/11 [00:00<00:05,  1.77it/s]
Reconstructing (incomplete total...):   1% 12.7k/1.92M [00:00<01:23, 22.9kB/s]
Reconstructing (incomplete total...):   1% 18.8k/3.55M [00:00<02:34, 22.9kB/s]
Reconstructing (incomplete total...):   0% 18.8k/5.66M [00:00<04:06, 22.9kB/s]
Reconstructing (incomplete total...):   0% 18.8k/7.49M [00:00<05:25, 22.9kB/s]

Fetching 11 files:  45% 5/11 [00:00<00:00,  8.82it/s]
Reconstructing (incomplete total...):  80% 7.49M/9.37M [00:00<00:00, 4.33MB/s]

Fetching 11 files:  82% 9/11 [00:00<00:00, 11.85it/s

## Stage checklist — CLOSED 2026-08-14 (one rerun pending)
- [x] Canonical pinned: aw-playworld:train/v1 (file sha 0f60f9b0, lineage-fp 042eb078)
- [x] a1v2 (269d0e, adapter 70c2ecb9) + b4v2 (c56ed2, adapter d4fcacdd) trained on identical bytes
- [x] c_v2_eval freeze gate 3cdcbc30 matched; evals a27857 / 7308ee
- [x] **RQ1 canonical: B4v2 vs A1v2 — 10/10 cells sig p=.0001** (ID pass +.227, adv +.263)
- [x] Sensitivity a1v2-vs-a1v1: all n.s. (D1 drift behaviorally benign; v1 result replicated)
- [x] x09h: b4v2 trunc/runaway 0.0; a1v2 trunc .994/runaway .888 (base-init pathology,
      A-track-wide; verdicts termination-insensitive — report as two-stage side benefit)
- [x] RERUN: b4v2-vs-b4v1 sensitivity (fetch fix applied) — expect ~null
- [x] RQ1 DONE → proceed: aw_08 B5 (parent b4v2 c56ed2) + A2v2 (parent a1v2 269d0e)


## B4v2 — retrain on the resolved frozen artifact (deviation D1 remediation)


In [ ]:
# @title x16 — resolve which frozen artifact A1 actually trained on (CPU)
A1_RUN_ID = "20260801-030335--a1-playworld-sft--s42--e24d72"
B4_RUN_ID = "20260812-235057--b4-playworld-sft-from-p1--s42--72f4f6"

!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1_RUN_ID}
!python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4_RUN_ID}

# List every persisted playworld_sft.jsonl copy here
!python scripts/x16_resolve_sft_provenance.py \
  --candidate m97j/aw-posttrain:train/v1/playworld_sft.jsonl \
  --candidate m97j/aw-playworld:preference_train/v1/playworld_sft.jsonl \
  --candidate m97j/aw-playworld:train/v1/playworld_sft.jsonl \
  --lineage runs/{A1_RUN_ID}/artifacts/lineage.json \
  --lineage runs/{B4_RUN_ID}/artifacts/lineage.json \
  --out runs/x16_sft_provenance.json


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0% 0/14 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/11.4M [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/186M [00:00<?, ?B/s] 
Reconstructing (incomplete total...):   0% 0.00/186M [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 577/186M [00:00<19:40:42, 2.63kB/s]
Reconstructing (incomplete total...):   0% 896/186M [00:00<19:40:42, 2.63kB/s]

Fetching 14 files:   7% 1/14 [00:00<00:03,  3.55it/s]
Reconstructing (incomplete total...):   0% 1.09k/186M [00:00<19:40:44, 2.63kB/s]
Reconstructing (incomplete total...):   0% 5.21k/186M [00:00<19:40:44, 2.63kB/s]
Reconstructing (incomplete total...):   0% 10.4k/186M [00:00<1:21:08, 38.2kB/s]
Reconstructing (incomplete total...):   0% 11.5k/186M [00:00<1:21:08, 38.2kB/s]
Reconstructing (incomplete total...):   0% 12.2k/186M [00:00<1:21:08, 38.2kB/s]
Reconstructing (incomplete total...):   0

## A1v2 + B4v2 — paired retrain on the pinned canonical artifact (D1 remediation, protocol v1.3)

Canonical: `m97j/aw-playworld:train/v1/playworld_sft.jsonl` (lineage-style
fingerprint sha256:042eb078...). Both arms retrain on the SAME bytes under
their original recipes — the RQ1 single variable (initialization) is restored
by construction rather than reconstruction.


In [ ]:
# @title g_v2_data — sha-pinned canonical SFT artifact (both arms)
CANONICAL_SHA = "0f60f9b043d826bfc998c3253925de1c781812d9d94905a4790d56e727a41f7f"  # paste DATASET_SHA256 printed on first run, then keep pinned

!python scripts/fetch_dataset.py \
  --repo m97j/aw-playworld --path train/v1/playworld_sft.jsonl \
  --output data/train/playworld_sft.jsonl --force \
  {"--expected-sha256 " + CANONICAL_SHA if CANONICAL_SHA else ""}


fetched dataset: hf://m97j/aw-playworld/train/v1/playworld_sft.jsonl
revision: main
materialized: data/train/playworld_sft.jsonl
dataset sha256: 0f60f9b043d826bfc998c3253925de1c781812d9d94905a4790d56e727a41f7f
DATASET_PATH=data/train/playworld_sft.jsonl
DATASET_SHA256=0f60f9b043d826bfc998c3253925de1c781812d9d94905a4790d56e727a41f7f


In [ ]:
# @title g_a1v2_retrain — A1 recipe, canonical data (control arm)
!python scripts/run_experiment.py \
  --config configs/experiments/a1_playworld_sft.yaml \
  --override experiment_name=a1v2-playworld-sft \
  --override data.source.local_path=data/train/playworld_sft.jsonl \
  --hf-sync-repo m97j/aw-runs-a1


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
run_id: 20260814-022256--a1v2-playworld-sft--s42--269d0e
config.json: 100% 729/729 [00:00<00:00, 8.71MB/s]
tokenizer_config.json: 100% 9.68k/9.68k [00:00<00:00, 8.04MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 109MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 73.0MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 288MB/s]
model.safetensors.index.json: 100% 32.9k/32.9k [00:00<00:00, 201MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/5.24G [00:00<?, 

In [ ]:
# @title g_b4v2_retrain — B4 recipe from B1v2 parent, canonical data (treatment arm)
B1V2_RUN_ID = "20260807-225109--b1-general-sft-v2--s42--e6e83b"
out = !python scripts/fetch_run.py --repo m97j/aw-runs-b1 --run-id {B1V2_RUN_ID}
b1v2_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
import json
b1v2_sha = json.load(open(f"runs/{B1V2_RUN_ID}/artifacts/lineage.json"))["output_adapter_sha256"]

!python scripts/run_experiment.py \
  --config configs/experiments/b4_playworld_sft_from_p1.yaml \
  --parent-adapter-dir {b1v2_dir} \
  --override lineage.parent_run_id={B1V2_RUN_ID} \
  --override lineage.parent_adapter.repo_id=m97j/aw-runs-b1 \
  --override lineage.parent_adapter.revision=main \
  --override lineage.parent_adapter.sha256={b1v2_sha} \
  --override experiment_name=b4v2-playworld-sft-from-p1 \
  --override data.source.local_path=data/train/playworld_sft.jsonl \
  --hf-sync-repo m97j/aw-runs-b4


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
run_id: 20260814-023603--b4v2-playworld-sft-from-p1--s42--c56ed2
Loading weights: 100% 399/399 [00:01<00:00, 334.47it/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
[transformers] warmup_ratio is de

In [ ]:
# @title c_v2_eval — both v2 arms on the frozen suites
A1V2_RUN_ID = "20260814-022256--a1v2-playworld-sft--s42--269d0e"  # <- from g_a1v2_retrain
B4V2_RUN_ID = "20260814-023603--b4v2-playworld-sft-from-p1--s42--c56ed2"  # <- from g_b4v2_retrain

!python scripts/build_eval_suites.py --episodes-per-suite 300
# freeze_fingerprint MUST equal sha256:3cdcbc30... (G3 gate) — abort otherwise

out = !python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1V2_RUN_ID}
a1v2_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
out = !python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4V2_RUN_ID}
b4v2_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {a1v2_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-a1
!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {b4v2_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-b4


eval_id: 300 episodes -> data/eval_suites/eval_id.jsonl (sha256:aceeea727d2b9eaed...)
eval_template_ood: 300 episodes -> data/eval_suites/eval_template_ood.jsonl (sha256:13580a6cbf7a4e566...)
eval_comp_ood: 300 episodes -> data/eval_suites/eval_comp_ood.jsonl (sha256:444191a244dcd77d1...)
eval_rule_ood: 300 episodes -> data/eval_suites/eval_rule_ood.jsonl (sha256:d73745108f5e7e207...)
eval_adversarial: 300 episodes -> data/eval_suites/eval_adversarial.jsonl (sha256:c73dd155acd069292...)

G3 freeze manifest -> data/eval_suites/freeze_manifest.json
Commit this manifest; training loaders must pass eval_family_ids as forbidden_family_ids (leakage gate).
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python

In [ ]:
# @title x09h + f_v2_analysis — canonical RQ1 table (B4v2 vs A1v2) + sensitivity
A1V2_EVAL = "20260814-025057--eval-playworld--s42--a27857"  # <- eval run ids printed by c_v2_eval
B4V2_EVAL = "20260814-032546--eval-playworld--s42--7308ee"

!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1V2_EVAL} --kind eval
!python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4V2_EVAL} --kind eval
!python scripts/x09_termination_audit.py \
  --run-dirs runs/{A1V2_EVAL} runs/{B4V2_EVAL} --out runs/x09_run_audit_v2arms.json

# RQ1 canonical
!python scripts/run_analysis.py \
  --run-a runs/{B4V2_EVAL} --label-a b4v2-two-stage \
  --run-b runs/{A1V2_EVAL} --label-b a1v2-direct \
  --output runs/{B4V2_EVAL}/analysis_b4v2_vs_a1v2.json --hf-sync-repo m97j/aw-runs-b4

# Sensitivity: each v2 arm vs its v1 (expected ~null if tie-break drift is benign)
A1_EVAL_V1 = "20260801-063425--eval-playworld--s42--3bf440"
B4_EVAL_V1 = "20260813-000613--eval-playworld--s42--c49f3a"
!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1_EVAL_V1} --kind eval
# v0.6.10 FIX: this fetch was missing -> FileNotFoundError on the b4v1 suites
!python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4_EVAL_V1} --kind eval
!python scripts/run_analysis.py \
  --run-a runs/{A1V2_EVAL} --label-a a1v2 --run-b runs/{A1_EVAL_V1} --label-b a1-v1 \
  --output runs/{A1V2_EVAL}/analysis_a1v2_vs_a1v1.json --hf-sync-repo m97j/aw-runs-a1
!python scripts/run_analysis.py \
  --run-a runs/{B4V2_EVAL} --label-a b4v2 --run-b runs/{B4_EVAL_V1} --label-b b4-v1 \
  --output runs/{B4V2_EVAL}/analysis_b4v2_vs_b4v1.json --hf-sync-repo m97j/aw-runs-b4


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0% 0/18 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/1.82M [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/3.90M [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/5.80M [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/5.81M [00:00<?, ?B/s]

Fetching 18 files:   6% 1/18 [00:00<00:03,  4.48it/s]
Reconstructing (incomplete total...):  53% 3.91M/7.39M [00:00<00:00, 8.07MB/s]
Reconstructing (incomplete total...):  79% 5.81M/7.40M [00:00<00:00, 8.07MB/s]
Reconstructing (incomplete total...):  61% 5.81M/9.45M [00:00<00:00, 8.07MB/s]
Reconstructing (incomplete total...):  78% 7.40M/9.45M [00:00<00:00, 8.07MB/s]
Reconstructing (incomplete total...):  78% 7.40M/9.46M [00:00<00:00, 20.7MB/s]
Reconstructing (incomplete total...):  64% 7.40M/11.5M [00:00<00:00, 20.7MB/s]
Reconstructing (incomplete total...):  55% 7.40M/13.4M [

## Champion standalone release — completed web Colab execution

The cells below retain the actual setup, verification, evaluation and publication
record. They are not instructions to restart the closed runtime.

Sequence: pinned fetch → FP32 merge/reload → FP32 task comparison → BF16 candidate
and task comparison → independent-control audit attachment → publication attempt
→ successful Xet-disabled retry. The read-only diagnostics cell preserves the
state observed when it was executed, even where later files were created.

The FP32 export passed merge and standalone checks. It is a disclosed precision
variant, not a claim of exact historical evaluator equivalence. Failed BF16,
review and upload observations remain visible alongside their resolution.

[Published revision](https://huggingface.co/m97j/aw-qwen3-8b-v1/tree/5b71e3e41d3c7630b4c6320aa67148ab653ac96d) ·
[Archived receipts and verification records](evidence/champion_release_20260923/README.md).


In [ ]:
# @title release diagnostics — read existing failure evidence only (no merge/upload)
import importlib.metadata

print("Python:", sys.version)
print("Interpreter:", sys.executable)
print("Workspace:", REPO_ROOT)
print("Output directory:", RELEASE_OUTPUT)
for package in ("torch", "transformers", "peft", "accelerate", "huggingface_hub", "safetensors"):
    try:
        print(package, importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError:
        print(package, "NOT INSTALLED")
for relative in ("plan.json", "verification/precision.json", "verification/merge.json", "verification/standalone.json",
                 "verified.json", "upload_started.json", "uploaded.json", "published.json"):
    evidence = REPO_ROOT / RELEASE_OUTPUT / relative
    print(f"\n[{relative}] exists={evidence.is_file()}")
    if evidence.is_file():
        record = json.loads(evidence.read_text(encoding="utf-8"))
        # Keep the full inventory in its file; show the decision-making evidence here.
        if relative == "verified.json":
            record = {key: value for key, value in record.items() if key != "files"}
        print(json.dumps(record, indent=2))
# If another prepare attempt is needed, explicitly choose a NEW output path first.
# RELEASE_OUTPUT = "runs/champion-v1-release-diagnostic-01"


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Interpreter: /usr/bin/python3
Workspace: /content/axiom-world-fp32
Output directory: runs/champion-v1-release-fp32
torch 2.11.0+cu128
transformers 5.13.1
peft 0.19.1
accelerate 1.14.0
huggingface_hub 1.23.0
safetensors 0.8.0

[plan.json] exists=False

[verification/precision.json] exists=False

[verification/merge.json] exists=False

[verification/standalone.json] exists=False

[verified.json] exists=False

[upload_started.json] exists=False

[uploaded.json] exists=False

[published.json] exists=False


In [ ]:
# @title publish_champion[b4v2] — pinned fetch and dry-run
release_command("scripts/common/fetch_run.py", "--repo", "m97j/aw-runs-b4",
    "--revision", "ef9a32095d5c4ed10fe4ab9b5914b320a89dd024",
    "--run-id", "20260814-023603--b4v2-playworld-sft-from-p1--s42--c56ed2",
    "--expected-sha256", "sha256:d4fcacddf21f758cdab904845ebdfee1eefde309c0edb6205bac64d5f07c76c8")
release_command("scripts/publish/v1/publish_champion.py", "--dry-run", "--output", RELEASE_OUTPUT)


[preflight] FP32 code=1daef869d312dfc91a6d400d56e7851a421eeee8; workspace=/content/axiom-world-fp32
[start] /usr/bin/python3 -u scripts/common/fetch_run.py --repo m97j/aw-runs-b4 --revision ef9a32095d5c4ed10fe4ab9b5914b320a89dd024 --run-id 20260814-023603--b4v2-playworld-sft-from-p1--s42--c56ed2 --expected-sha256 sha256:d4fcacddf21f758cdab904845ebdfee1eefde309c0edb6205bac64d5f07c76c8
[log] /content/axiom-world-release-logs/20260923T144559Z-f4626c87.log
fetched from hf://m97j/aw-runs-b4 (sha256 verified: sha256:d4fcacddf21f...)
ADAPTER_DIR=runs/20260814-023603--b4v2-playworld-sft-from-p1--s42--c56ed2/artifacts/final_adapter
[exit] 0; log=/content/axiom-world-release-logs/20260923T144559Z-f4626c87.log
[preflight] FP32 code=1daef869d312dfc91a6d400d56e7851a421eeee8; workspace=/content/axiom-world-fp32
[start] /usr/bin/python3 -u scripts/publish/v1/publish_champion.py --dry-run --output runs/champion-v1-release-fp32
[log] /content/axiom-world-release-logs/20260923T144614Z-8c2309da.log
Champ

In [ ]:
# @title publish_champion[b4v2] — local FP32 merge and fresh-process verification
if not callable(globals().get("verify_release_checkout")):
    raise RuntimeError("Old kernel header detected. Run the updated common header before prepare.")
verify_release_checkout()
if not RELEASE_OUTPUT.startswith("runs/champion-v1-release-fp32"):
    raise RuntimeError("Unexpected output path: review the FP32 header configuration before prepare")
release_command("scripts/publish/v1/publish_champion.py", "--prepare",
                "--dtype", "float32", "--device", "cuda", "--output", RELEASE_OUTPUT)
receipt = json.loads((REPO_ROOT / RELEASE_OUTPUT / "verified.json").read_text())
print(json.dumps(receipt["verification"], indent=2))
release_command("scripts/publish/v1/publish_champion.py", "--dry-run", "--output", RELEASE_OUTPUT)


[preflight] FP32 code=1daef869d312dfc91a6d400d56e7851a421eeee8; workspace=/content/axiom-world-fp32
[preflight] FP32 code=1daef869d312dfc91a6d400d56e7851a421eeee8; workspace=/content/axiom-world-fp32
[start] /usr/bin/python3 -u scripts/publish/v1/publish_champion.py --prepare --dtype float32 --device cuda --output runs/champion-v1-release-fp32
[log] /content/axiom-world-release-logs/20260923T144618Z-49feb2cd.log
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensu

In [ ]:
# @title compare original adapter and FP32 export — 1500 frozen tasks each, no upload
# Requires the existing VERIFIED_LOCAL_RELEASE. Do not rerun prepare for this step.
# This uses GPU inference; preserve partial output and use a new path after investigating failures.
COMPARISON_OUTPUT = "runs/champion-v1-comparison"
release_command("scripts/publish/v1/compare_champion.py",
                "--release", RELEASE_OUTPUT, "--output", COMPARISON_OUTPUT,
                "--batch-size", "4")
comparison = json.loads((REPO_ROOT / COMPARISON_OUTPUT / "comparison.json").read_text())
print(json.dumps(comparison["suites"], indent=2))
print("Task comparison complete; publication still requires review.")


[preflight] FP32 code=1daef869d312dfc91a6d400d56e7851a421eeee8; workspace=/content/axiom-world-fp32
[start] /usr/bin/python3 -u scripts/publish/v1/compare_champion.py --release runs/champion-v1-release-fp32 --output runs/champion-v1-comparison --batch-size 4
[log] /content/axiom-world-release-logs/20260923T145155Z-932abd24.log
[preflight] All 1500 task prompt token sequences match
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to T

### BF16 candidate audit

The corrected conversion preserves native non-persistent RoPE buffers.
The following task evaluation completed all 1500 episodes in both arms at batch 256.
Its subsequent combined-review step failed because the earlier FP32 run used batch 4.
This traceback does not mean the BF16 task evaluation was incomplete.


In [ ]:
# @title derive BF16 candidate — preserve verified FP32 source, no upload
release_command("scripts/publish/v1/derive_bf16_candidate.py",
                "--source", RELEASE_OUTPUT, "--output", BF16_RELEASE_OUTPUT,
                "--device", "cuda")
receipt = json.loads((REPO_ROOT / BF16_RELEASE_OUTPUT / "verified.json").read_text())
print(json.dumps(receipt["verification"], indent=2))


[preflight] FP32 code=5298f52b3f60aaed8cb003e4284d53ea53cc263f; workspace=/content/axiom-world-fp32
[start] /usr/bin/python3 -u scripts/publish/v1/derive_bf16_candidate.py --source runs/champion-v1-release-fp32 --output runs/champion-v1-release-bf16-candidate-native-buffers --device cuda
[log] /content/axiom-world-release-logs/20260923T163201Z-4bc419d2.log
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
[cast probe] FP32 source
[cast probe] BF16 candidate
{
  "fp32_vs_bf16": {
    "max_abs": 11.935070037841797,
    "mean_abs": 0.8374203155665569,
    "top1_agreement": 0.9228855721393034,
    "generation_agreement": 0.125,
    "pas

In [ ]:
# @title compare BF16 and review precision — 1500 frozen tasks per arm, no upload
release_command("scripts/publish/v1/compare_champion.py",
                "--release", BF16_RELEASE_OUTPUT, "--output", BF16_COMPARISON_OUTPUT,
                "--batch-size", "256")
release_command("scripts/publish/v1/review_precision.py",
                "--fp32-comparison", COMPARISON_OUTPUT,
                "--bf16-comparison", BF16_COMPARISON_OUTPUT)
print("Review precision_review.json before choosing the final deployment precision.")


[preflight] FP32 code=5298f52b3f60aaed8cb003e4284d53ea53cc263f; workspace=/content/axiom-world-fp32
[start] /usr/bin/python3 -u scripts/publish/v1/compare_champion.py --release runs/champion-v1-release-bf16-candidate-native-buffers --output runs/champion-v1-comparison-bf16-native-buffers --batch-size 256
[log] /content/axiom-world-release-logs/20260923T163531Z-58669c86.log
[preflight] All 1500 task prompt token sequences match
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the ada

RuntimeError: Command failed (exit=1). See the traceback above or /content/axiom-world-release-logs/20260923T165658Z-7affabaf.log

### FP32 deployment decision and evidence attachment

BF16 degraded against its own adapter control. The completed reports were reviewed
as independent-control runs, not treated as a matched precision comparison.
The following CPU-only finalization attached both runs and their limitations to
the FP32 model card while preserving the verified model weights.


In [ ]:
# @title finalize FP32 task audit — completed reports only, no GPU and no upload
release_command("scripts/publish/v1/finalize_fp32_audit.py",
                "--release", RELEASE_OUTPUT,
                "--fp32-comparison", COMPARISON_OUTPUT,
                "--bf16-comparison", BF16_COMPARISON_OUTPUT)
release_command("scripts/publish/v1/publish_champion.py", "--dry-run", "--output", RELEASE_OUTPUT)
print("Audit attached. Review FP32 disclosure, then enable the final publication switches.")


[preflight] FP32 code=e2a66ab73c4054e59339b8870bf437a0b586141f; workspace=/content/axiom-world-fp32
[start] /usr/bin/python3 -u scripts/publish/v1/finalize_fp32_audit.py --release runs/champion-v1-release-fp32 --fp32-comparison runs/champion-v1-comparison --bf16-comparison runs/champion-v1-comparison-bf16-native-buffers
[log] /content/axiom-world-release-logs/20260923T170923Z-64e87d22.log
{
  "selected_precision": "float32",
  "selection_basis": "Retain independently verified FP32 export; reject BF16 after within-run regressions. Not a matched FP32-versus-BF16 superiority claim or historical equivalence claim.",
  "source_receipt_sha256": "6312c9112d93090d353f3b2e522093c9fbc72322e6c718af50f7e6423d728a18",
  "review": {
    "status": "completed_requires_review",
    "reference_repeated_exactly": false,
    "suites": {
      "eval_adversarial": {
        "fp32": {
          "episodes": 300,
          "reference_pass": 254,
          "candidate_pass": 260,
          "regressions": 3,
    

In [13]:
# @title publish_champion[b4v2] — explicit HF publication after reviewing verification
# This cell changes the public HF repo; it does not push GitHub or start training.
# Set True only after reviewing precision.json and the disclosed FP32 model card.
receipt = json.loads((REPO_ROOT / RELEASE_OUTPUT / "verified.json").read_text())
decision = json.loads((REPO_ROOT / RELEASE_OUTPUT / "model/evaluation/precision_audit/decision.json").read_text())
if (decision.get("selected_precision") != "float32"
        or decision["source_receipt_sha256"] != receipt.get("task_audit_source_receipt_sha256")):
    raise RuntimeError("Missing bound FP32 task-audit finalization")
comparison = json.loads((REPO_ROOT / COMPARISON_OUTPUT / "comparison.json").read_text())
receipt_hash = receipt["task_audit_source_receipt_sha256"]
if (comparison.get("status") != "completed_requires_review"
        or comparison["request"]["release_receipt_sha256"] != receipt_hash):
    raise RuntimeError("Missing task comparison for this exact release")
PUBLISH_TASK_COMPARISON_REVIEWED = True
if not PUBLISH_TASK_COMPARISON_REVIEWED:
    raise RuntimeError("Review paired task performance before publication")
PUBLISH_PRECISION_VARIANT = True
if not PUBLISH_PRECISION_VARIANT:
    raise RuntimeError("Review the FP32 variant and historical precision comparison before publication")
release_command("scripts/publish/v1/publish_champion.py", "--publish", "--accept-precision-change", "--output", RELEASE_OUTPUT)
print((REPO_ROOT / RELEASE_OUTPUT / "published.json").read_text())
# Save this local notebook in VS Code and retain the runtime receipts before disconnecting.


[preflight] FP32 code=e2a66ab73c4054e59339b8870bf437a0b586141f; workspace=/content/axiom-world-fp32
[start] /usr/bin/python3 -u scripts/publish/v1/publish_champion.py --publish --accept-precision-change --output runs/champion-v1-release-fp32
[log] /content/axiom-world-release-logs/20260923T171059Z-dfc463f8.log
{
  "target_repo": "m97j/aw-qwen3-8b-v1",
  "expected_head": "f33d2d16125e89eb38d4b668a2c20a6929ad3784",
  "legacy_revision": "f33d2d16125e89eb38d4b668a2c20a6929ad3784",
  "legacy_tag": "protocol-v1-adapter",
  "delete_paths": [
    "adapter_config.json",
    "adapter_model.safetensors"
  ],
  "history_rewrite": false
}
Traceback (most recent call last):
  File "/content/axiom-world-fp32/scripts/publish/v1/publish_champion.py", line 294, in <module>
    sys.exit(main())
             ~~~~^^
  File "/content/axiom-world-fp32/scripts/publish/v1/publish_champion.py", line 284, in main
    publish(args.output, execute=args.publish, accept_precision_change=args.accept_precision_change)

RuntimeError: Command failed (exit=1). See the traceback above or /content/axiom-world-release-logs/20260923T171059Z-dfc463f8.log

### Successful upload recovery

The preceding publication attempt timed out in the Xet upload transport. Remote HEAD
was checked before retry, and the failed attempt marker was archived. The following
cell disabled Xet for the new subprocess and completed publication. Preserve both
attempts; neither cell should be rerun against the already published release.


In [14]:
# @title retry publication after verified Xet timeout
from huggingface_hub import HfApi

output = REPO_ROOT / RELEASE_OUTPUT
receipt = json.loads((output / "verified.json").read_text())
marker = output / "upload_started.json"
archive = output / "upload_started.xet-timeout-20260923T171059Z.json"

if any((output / name).exists() for name in ("uploaded.json", "published.json")):
    raise RuntimeError("Upload receipt exists; inspect remote state before retrying")
if not marker.exists() or archive.exists():
    raise RuntimeError("Unexpected retry state; preserve files and inspect")
if json.loads(marker.read_text()) != receipt["plan"]:
    raise RuntimeError("Upload marker differs from verified release plan")

# Revalidate local payload and remote migration plan without uploading.
release_command("scripts/publish/v1/publish_champion.py",
                "--dry-run", "--output", RELEASE_OUTPUT)

head = HfApi().repo_info(
    receipt["plan"]["target_repo"], repo_type="model", revision="main"
).sha
if head != receipt["plan"]["expected_head"]:
    raise RuntimeError("Remote HEAD changed; do not retry blindly")

marker.rename(archive)  # Preserve the failed attempt; do not delete it.
os.environ["HF_HUB_DISABLE_XET"] = "1"

release_command("scripts/publish/v1/publish_champion.py",
                "--publish", "--accept-precision-change",
                "--output", RELEASE_OUTPUT)
print((output / "published.json").read_text())

[preflight] FP32 code=e2a66ab73c4054e59339b8870bf437a0b586141f; workspace=/content/axiom-world-fp32
[start] /usr/bin/python3 -u scripts/publish/v1/publish_champion.py --dry-run --output runs/champion-v1-release-fp32
[log] /content/axiom-world-release-logs/20260923T172347Z-644ff831.log
{
  "target_repo": "m97j/aw-qwen3-8b-v1",
  "expected_head": "f33d2d16125e89eb38d4b668a2c20a6929ad3784",
  "legacy_revision": "f33d2d16125e89eb38d4b668a2c20a6929ad3784",
  "legacy_tag": "protocol-v1-adapter",
  "delete_paths": [
    "adapter_config.json",
    "adapter_model.safetensors"
  ],
  "history_rewrite": false
}
Dry run: verified payload; no remote writes
[exit] 0; log=/content/axiom-world-release-logs/20260923T172347Z-644ff831.log
[preflight] FP32 code=e2a66ab73c4054e59339b8870bf437a0b586141f; workspace=/content/axiom-world-fp32
[start] /usr/bin/python3 -u scripts/publish/v1/publish_champion.py --publish --accept-precision-change --output runs/champion-v1-release-fp32
[log] /content/axiom-world-r

## Appendix — earlier precision diagnostics (historical)

These cells are retained as diagnostic evidence, not continuation steps. Their
`/content/axiom-world` paths and kernel variables refer to an earlier checkout.
Execution order cannot be inferred from notebook position or reset execution counts.
The older whole-model BF16 cast also rounded non-persistent RoPE buffers; its
measurements must not be interpreted as isolated weight-rounding effects.


In [ ]:
from pathlib import Path
import json

report_path = (
    Path("/content/axiom-world")
    / "runs/champion-v1-release"
    / "verification/merge.json"
)
report = json.loads(report_path.read_text(encoding="utf-8"))
print(json.dumps(report, indent=2, ensure_ascii=False))

limits = {
    "max_abs": ("<=", 0.5),
    "mean_abs": ("<=", 0.03),
    "top1_agreement": (">=", 0.99),
    "generation_agreement": (">=", 0.90),
}
for name, (operator, limit) in limits.items():
    value = report[name]
    passed = value <= limit if operator == "<=" else value >= limit
    print(f"{name}: {value} {operator} {limit} → {'PASS' if passed else 'FAIL'}")

{
  "max_abs": 11.125,
  "mean_abs": 0.8322163692142232,
  "top1_agreement": 0.917910447761194,
  "generation_agreement": 0.28125,
  "passed": false,
  "runtime": {
    "device": "cuda",
    "torch_cuda": "12.8",
    "gpu_name": "NVIDIA RTX PRO 6000 Blackwell Server Edition",
    "attention": "sdpa",
    "reference": "PEFT default adapter dtype promotion"
  }
}
max_abs: 11.125 <= 0.5 → FAIL
mean_abs: 0.8322163692142232 <= 0.03 → FAIL
top1_agreement: 0.917910447761194 >= 0.99 → FAIL
generation_agreement: 0.28125 >= 0.9 → FAIL


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("/content/axiom-world")
SRC_ROOT = REPO_ROOT / "src"

if not (SRC_ROOT / "axiom_world/models/champion_release.py").is_file():
    raise FileNotFoundError(
        f"Project source not found: {SRC_ROOT}\n"
        "First verify that the notebook header completed the repository clone."
    )

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

In [ ]:
# @title merge diagnosis — precision stages, 4 probes, no publication
import gc
import json
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from axiom_world.models.champion_release import (
    _probe, _check_saved_adapter, _check_saved_modules, compare_logits,
)

request = json.loads(
    (Path(RELEASE_OUTPUT) / "verification/request.json")
    .read_text(encoding="utf-8")
)

def diagnose_merge(request):
    all_probes = request["probes"]
    indices = [0, len(all_probes) // 4,
               len(all_probes) // 2, 3 * len(all_probes) // 4]
    probes = [all_probes[i] for i in indices]
    adapter = Path(request["adapter"])

    tokenizer = AutoTokenizer.from_pretrained(
        adapter, local_files_only=True, trust_remote_code=False,
    )
    base = AutoModelForCausalLM.from_pretrained(
        request["base"], revision=request["revision"],
        dtype=torch.bfloat16, device_map=request["device"],
        attn_implementation="sdpa", trust_remote_code=False,
        local_files_only=True,
    )
    model = PeftModel.from_pretrained(
        base, adapter, is_trainable=False,
    ).eval()
    _check_saved_adapter(model, adapter)

    model.generation_config.eos_token_id = list(dict.fromkeys([
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|im_end|>"),
    ]))
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.do_sample = False

    print("[1/4] Original BF16-base PEFT reference", flush=True)
    original = _probe(model, tokenizer, probes)

    print("[2/4] Same loaded model promoted to FP32", flush=True)
    model.float()
    fp32_adapter = _probe(model, tokenizer, probes)

    print("[3/4] Merge in FP32", flush=True)
    merged = model.merge_and_unload(
        safe_merge=True, progressbar=False,
    ).eval()
    _check_saved_modules(merged, adapter)
    fp32_merged = _probe(merged, tokenizer, probes)

    print("[4/4] Cast merged model to BF16", flush=True)
    merged.to(dtype=torch.bfloat16)
    _check_saved_modules(merged, adapter)
    bf16_merged = _probe(merged, tokenizer, probes)

    def comparison(left, right):
        return compare_logits(left[0], right[0], left[1], right[1])

    input_weight = merged.get_input_embeddings().weight
    output_weight = merged.get_output_embeddings().weight
    return {
        "diagnostic_only": True,
        "probe_indices": indices,
        "original_vs_fp32_adapter": comparison(original, fp32_adapter),
        "fp32_adapter_vs_fp32_merged": comparison(fp32_adapter, fp32_merged),
        "fp32_merged_vs_bf16_merged": comparison(fp32_merged, bf16_merged),
        "original_vs_bf16_merged": comparison(original, bf16_merged),
        "merged_embeddings": {
            "config_tie_word_embeddings": merged.config.tie_word_embeddings,
            "same_storage": (
                input_weight.data_ptr() == output_weight.data_ptr()
            ),
            "equal_values": torch.equal(input_weight, output_weight),
        },
    }

# Disable TF32 for FP32 diagnostics and restore the original settings afterward.
previous_precision = torch.get_float32_matmul_precision()
try:
    torch.set_float32_matmul_precision("highest")
    diagnosis = diagnose_merge(request)
    print(json.dumps(diagnosis, indent=2, ensure_ascii=False))
finally:
    torch.set_float32_matmul_precision(previous_precision)
    gc.collect()
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


[1/4] Original BF16-base PEFT reference
[2/4] Same loaded model promoted to FP32
[3/4] Merge in FP32
[4/4] Cast merged model to BF16
{
  "diagnostic_only": true,
  "probe_indices": [
    0,
    8,
    16,
    24
  ],
  "original_vs_fp32_adapter": {
    "max_abs": 1.8473002910614014,
    "mean_abs": 0.19291247957797422,
    "top1_agreement": 0.9545454545454546,
    "generation_agreement": 0.5,
    "passed": false
  },
  "fp32_adapter_vs_fp32_merged": {
    "max_abs": 0.0001125335693359375,
    "mean_abs": 7.477118867998184e-06,
    "top1_agreement": 1.0,
    "generation_agreement": 1.0,
    "passed": true
  },
  "fp32_merged_vs_bf16_merged": {
    "max_abs": 10.420270919799805,
    "mean_abs": 1.03218556348823,
    "top1_agreement": 0.8181818181818182,
    "generation_agreement": 0.5,
    "passed": false
  },
  "original_vs_bf16_merged": {
    "max_abs": 10.03125,
    "mean_abs": 0.9541886656793964,
    "top1_agreement": 0.8409090909090909,
    "generation_agreement": 0.25,
    "passed"

## Appendix — runtime evidence download

This is the original export cell and its output. The supplied release ZIP contains
receipts and selected verification files, **not** full model weights, per-episode
comparison traces, or probe-logit tensors. The published HF revision contains the
model and attached task-audit evidence. See the local evidence README for inventory.


In [19]:
import shutil
from google.colab import files

# 1. Define the source directories and archive filenames.
folder_paths = ['/content/axiom-world-release-logs', '/content/axiom-world-fp32/runs/champion-v1-release-fp32']
zip_file_names = ['/content/axiom-world-release-logs.zip', '/content/axiom-world-fp32/runs/champion-v1-release-fp32.zip']

# 2. Archive the directories as ZIP files.
for folder_path in folder_paths:
  shutil.make_archive(folder_path, 'zip', folder_path)
  print(f"zipped {folder_path}")

# 3. Download the archives to the local computer.
for zip_file_name in zip_file_names:
  files.download(zip_file_name)


zipped /content/axiom-world-release-logs
zipped /content/axiom-world-fp32/runs/champion-v1-release-fp32


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>